In [106]:
import pandas
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import numpy as np
import xgboost as xgb
from xgboost.callback import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import math

In [107]:
data = pandas.read_csv("../new_code/DATASET.csv")

In [108]:
data.head()

,Current,Voltage,Ah Out,Cumulative Actual Disch Ah,Power,Remaining Capacity,Time to Depletion,type,capacity,charged
0,9.36,11.84,0.156000,0.156000,110.8224,26.844000,10324.615385,b2,88.81,27.0
1,9.34,11.84,0.155667,0.311667,110.5856,26.688333,10286.723769,b2,88.81,27.0
2,9.34,11.83,0.155667,0.467333,110.4922,26.532667,10226.723769,b2,88.81,27.0
3,7.14,11.88,0.119000,0.586333,84.8232,26.413667,13317.815126,b2,88.81,27.0
4,7.13,11.88,0.118833,0.705167,84.7044,26.294833,13276.493689,b2,88.81,27.0


In [109]:
data.shape

(5437, 10)

In [110]:
TARGET_VARIABLE = 'Time to Depletion'

In [111]:
Y = data[TARGET_VARIABLE]
X = data.drop(TARGET_VARIABLE, axis=1)

numerical_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

print("Numerical Featrures are : ", numerical_features)
print("Categorical Featrures are : ", categorical_features)

Numerical Featrures are :  ['Current', 'Voltage', 'Ah Out', 'Cumulative Actual Disch Ah', 'Power', 'Remaining Capacity', 'capacity', 'charged']
Categorical Featrures are :  ['type']


In [112]:
numerical_transformer = Pipeline(steps=[
    ('pass',
     'passthrough')
])
categorical_transformer = Pipeline(steps=[
    ('onehot',
     OneHotEncoder(handle_unknown='ignore',
                   sparse_output=False))
])

In [113]:
preprocessor = ColumnTransformer(transformers=[
    ('num', numerical_transformer, numerical_features),
    ('cat', categorical_transformer, categorical_features)
]
    ,remainder='passthrough')

In [114]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.15, random_state=42)

In [115]:
type(X_train)

pandas.core.frame.DataFrame

In [116]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [117]:
X_test

,Current,Voltage,Ah Out,Cumulative Actual Disch Ah,Power,Remaining Capacity,type,capacity,charged
3204,8.47,10.62,0.141167,85.515000,89.9514,-0.515000,b2,85.00,85.00
4515,7.50,12.33,0.125000,16.162458,92.4750,68.837542,b5,85.00,85.00
2146,9.51,11.79,0.158500,44.043500,112.1229,44.766500,b2,88.81,88.81
3105,24.11,11.30,0.401833,63.895833,272.4430,21.104167,b2,85.00,85.00
881,17.42,11.77,0.290333,26.487167,205.0334,54.792833,b1,81.28,81.28
...,...,...,...,...,...,...,...,...,...
4333,8.32,11.09,0.138667,27.725167,92.2688,8.274833,b1,81.84,36.00
3254,17.36,12.28,0.289333,1.066167,213.1808,87.283833,b3,88.35,88.35
2357,4.41,9.56,0.073500,85.251833,42.1596,3.558167,b2,88.81,88.81
4990,1.65,10.70,0.027500,81.538292,17.6550,3.461708,b5,85.00,85.00


In [118]:
X_test_processed

array([[ 8.47      , 10.62      ,  0.14116667, ...,  0.        ,
         0.        ,  0.        ],
       [ 7.5       , 12.33      ,  0.125     , ...,  0.        ,
         1.        ,  0.        ],
       [ 9.51      , 11.79      ,  0.1585    , ...,  0.        ,
         0.        ,  0.        ],
       ...,
       [ 4.41      ,  9.56      ,  0.0735    , ...,  0.        ,
         0.        ,  0.        ],
       [ 1.65      , 10.7       ,  0.0275    , ...,  0.        ,
         1.        ,  0.        ],
       [ 1.75      , 10.35      ,  0.02916667, ...,  0.        ,
         1.        ,  0.        ]])

In [119]:
xgb_model = xgb.XGBRegressor(
    objective='reg:squarederror',
    n_estimators=100,
    learning_rate=0.1,
    max_depth=10,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='mae'
)

In [120]:
early_stopping = EarlyStopping(
    rounds=10,
    metric_name='mae',
    save_best=True,
)

In [121]:
eval_set = [(X_test_processed, Y_test)]
xgb_model.set_params(callbacks=[early_stopping])
xgb_model.fit(X_train_processed, Y_train,
              eval_set=eval_set,
              verbose=False)

XGBRegressor(base_score=None, booster=None,
             callbacks=[<xgboost.callback.EarlyStopping object at 0x00000222DE0D7670>],
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric='mae', feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.1, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=10,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=100,
             n_jobs=None, num_parallel_tree=None, ...)

In [122]:
Y_pred = xgb_model.predict(X_test_processed)

In [123]:
mae = mean_absolute_error(Y_test, Y_pred)

print(f"Mean Absolute Error is : {mae:.2f}")

Mean Absolute Error is : 160.49


In [124]:
xgb_model.save_model("../models/battery_xgboost_model.json")

In [125]:
loaded_model = xgb.XGBRegressor()
loaded_model.load_model("../models/battery_xgboost_model.json")

In [126]:
y_loaded_pred = loaded_model.predict(X_test_processed)

mae_loaded = mean_absolute_error(Y_test, y_loaded_pred)
print(f"Mean Absolute Error is : {mae_loaded:.2f}")

Mean Absolute Error is : 160.49
